# Clustering Crédit - Partie 3: Interprétation Business & Marketing

🎯 **Objectif**: Transformer les clusters en segments actionnables avec des recommandations marketing

📋 **Ce que vous allez faire**:
1. Choisir le meilleur algorithme de clustering
2. Profiler chaque segment (personas détaillés)
3. Définir des actions marketing par segment
4. Estimer l'impact business potentiel
5. Créer un plan d'action concret

💡 **Rôle**: Vous présentez vos résultats au Directeur Marketing pour validation et mise en œuvre.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown, SelectMultiple, interactive
import warnings
warnings.filterwarnings('ignore')

plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)

print("🚀 Environnement prêt pour l'analyse business!")

## 📂 Étape 1: Récupération des résultats de clustering

Chargeons les données et reproduisons le meilleur clustering de la Partie 2.

In [ ]:
# Chargement des données (avec fallback si Partie 1 non disponible)
try:
    X_raw = pd.read_csv('donnees_clustering_partie1.csv')
    print("✅ Données chargées depuis la Partie 1")
except FileNotFoundError:
    print("⚠️ Création d'un jeu de données de démonstration...")
    np.random.seed(42)
    n_samples = 800
    X_raw = pd.DataFrame({
        'AGE': np.random.normal(35, 10, n_samples),
        'utilization': np.random.beta(2, 5, n_samples),
        'repay_ratio_recent': np.random.beta(3, 2, n_samples),
        'nb_retards': np.random.poisson(1.2, n_samples),
        'facture_moyenne': np.random.lognormal(8, 1, n_samples),
        'paiement_moyen': np.random.lognormal(7.5, 1.2, n_samples)
    })

# Standardisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f"📏 Données prêtes: {X_raw.shape}")
display(X_raw.head())

## 🏆 Étape 2: Sélection Interactive du Meilleur Modèle

Comparez les algorithmes et choisissez celui qui convient le mieux à votre cas d'usage.

In [ ]:
@interact(
    algorithme=Dropdown(options=['K-Means', 'CAH (Ward)', 'DBSCAN'], value='K-Means', description='Algorithme:'),
    k_ou_eps=FloatSlider(min=2, max=10, step=1, value=4, description='K ou eps:'),
    afficher_metriques=Dropdown(options=['Oui', 'Non'], value='Oui', description='Métriques:')
)
def choisir_modele(algorithme, k_ou_eps, afficher_metriques):
    """Sélection interactive du modèle final"""
    
    if algorithme == 'K-Means':
        model = KMeans(n_clusters=int(k_ou_eps), random_state=42, n_init=10)
        labels = model.fit_predict(X_scaled)
        titre = f'K-Means (K={int(k_ou_eps)})'
        
    elif algorithme == 'CAH (Ward)':
        model = AgglomerativeClustering(n_clusters=int(k_ou_eps), linkage='ward')
        labels = model.fit_predict(X_scaled)
        titre = f'CAH Ward (K={int(k_ou_eps)})'
        
    else:  # DBSCAN
        model = DBSCAN(eps=k_ou_eps, min_samples=5)
        labels = model.fit_predict(X_scaled)
        titre = f'DBSCAN (eps={k_ou_eps:.1f})'
    
    # Visualisation
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    scatter = plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels, cmap='viridis', alpha=0.7)
    plt.xlabel(X_raw.columns[0])
    plt.ylabel(X_raw.columns[1])
    plt.title(titre)
    plt.colorbar(scatter)
    
    plt.subplot(1, 2, 2)
    unique, counts = np.unique(labels, return_counts=True)
    colors = ['gray' if u == -1 else None for u in unique]
    plt.bar([str(u) for u in unique], counts, color=colors)
    plt.title('Taille des segments')
    plt.xlabel('Segment')
    plt.ylabel('Nombre de clients')
    
    plt.tight_layout()
    plt.show()
    
    # Métriques et conseils
    if afficher_metriques == 'Oui':
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = list(labels).count(-1) if -1 in labels else 0
        
        print(f"📊 Résultats {algorithme}:")
        print(f"  • Segments créés: {n_clusters}")
        if n_noise > 0:
            print(f"  • Points de bruit: {n_noise} ({n_noise/len(labels):.1%})")
        
        if n_clusters > 1:
            try:
                sil = silhouette_score(X_scaled, labels)
                print(f"  • Silhouette: {sil:.3f}")
                
                if sil > 0.5:
                    print("  ✅ Excellente séparation des segments")
                elif sil > 0.3:
                    print("  ✅ Bonne séparation des segments")
                else:
                    print("  ⚠️ Séparation faible, considérez d'autres paramètres")
            except:
                print("  • Silhouette: Non calculable")
        
        # Conseils business
        print(f"\n💡 Conseils business:")
        if n_clusters <= 3:
            print("  • Segments larges → Stratégies générales mais moins personnalisées")
        elif n_clusters <= 6:
            print("  • Nombre optimal → Bon équilibre personnalisation/opérationnalité")
        else:
            print("  • Nombreux segments → Très personnalisé mais complexe à gérer")
    
    # Sauvegarde pour les étapes suivantes
    global labels_final, modele_final
    labels_final = labels
    modele_final = algorithme
    
    return labels

# Application par défaut (K-Means K=4)
labels_final = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(X_scaled)
modele_final = 'K-Means'

## 👥 Étape 3: Profiling Détaillé des Segments

Analysons en détail chaque segment pour créer des personas.

In [ ]:
# Création du dataframe avec segments
df_segments = X_raw.copy()
df_segments['segment'] = labels_final

@interact(
    segment=Dropdown(options=[f'Segment {i}' for i in sorted(set(labels_final)) if i != -1], 
                    value='Segment 0', description='Segment:'),
    vue=Dropdown(options=['Profil complet', 'Comparaison', 'Distribution'], 
                value='Profil complet', description='Vue:')
)
def analyser_segment(segment, vue):
    """Analyse détaillée d'un segment"""
    seg_id = int(segment.split()[-1])
    seg_data = df_segments[df_segments['segment'] == seg_id]
    
    if len(seg_data) == 0:
        print(f"⚠️ Aucun client dans le {segment}")
        return
    
    print(f"📊 {segment} - {len(seg_data)} clients ({len(seg_data)/len(df_segments):.1%} de la base)")
    
    if vue == 'Profil complet':
        # Statistiques détaillées
        stats = seg_data.describe().round(2)
        display(stats)
        
        # Caractéristiques principales
        print(f"\n🎯 Caractéristiques principales:")
        
        # Age
        age_moy = seg_data['AGE'].mean()
        if age_moy < 30:
            print(f"  • Jeunes clients (âge moyen: {age_moy:.0f} ans)")
        elif age_moy > 50:
            print(f"  • Clients matures (âge moyen: {age_moy:.0f} ans)")
        else:
            print(f"  • Clients d'âge moyen ({age_moy:.0f} ans)")
        
        # Utilisation
        if 'utilization' in seg_data.columns:
            util_moy = seg_data['utilization'].mean()
            if util_moy > 0.7:
                print(f"  • Forte utilisation du crédit ({util_moy:.1%})")
            elif util_moy < 0.3:
                print(f"  • Faible utilisation du crédit ({util_moy:.1%})")
            else:
                print(f"  • Utilisation modérée du crédit ({util_moy:.1%})")
        
        # Remboursement
        if 'repay_ratio_recent' in seg_data.columns:
            repay_moy = seg_data['repay_ratio_recent'].mean()
            if repay_moy > 0.8:
                print(f"  • Excellents payeurs (ratio: {repay_moy:.1%})")
            elif repay_moy < 0.5:
                print(f"  • Remboursements irréguliers (ratio: {repay_moy:.1%})")
            else:
                print(f"  • Remboursements corrects (ratio: {repay_moy:.1%})")
        
        # Retards
        if 'nb_retards' in seg_data.columns:
            retards_moy = seg_data['nb_retards'].mean()
            if retards_moy > 2:
                print(f"  • Historique de retards fréquents ({retards_moy:.1f} mois)")
            elif retards_moy < 0.5:
                print(f"  • Très peu de retards ({retards_moy:.1f} mois)")
            else:
                print(f"  • Retards occasionnels ({retards_moy:.1f} mois)")
    
    elif vue == 'Comparaison':
        # Comparaison avec la moyenne générale
        print(f"\n📈 Comparaison avec la moyenne générale:")
        
        for col in X_raw.columns:
            seg_mean = seg_data[col].mean()
            global_mean = df_segments[col].mean()
            ratio = seg_mean / global_mean if global_mean != 0 else 1
            
            if ratio > 1.2:
                trend = f"↑ +{(ratio-1)*100:.0f}%"
            elif ratio < 0.8:
                trend = f"↓ {(1-ratio)*100:.0f}%"
            else:
                trend = "≈ similaire"
            
            print(f"  • {col}: {seg_mean:.2f} vs {global_mean:.2f} {trend}")
    
    elif vue == 'Distribution':
        # Graphiques de distribution
        n_cols = len(X_raw.columns)
        fig, axes = plt.subplots(2, 3, figsize=(15, 8))
        axes = axes.flatten()
        
        for i, col in enumerate(X_raw.columns[:6]):
            if i < len(axes):
                # Histogramme du segment vs global
                axes[i].hist(df_segments[col], bins=20, alpha=0.5, label='Tous', density=True)
                axes[i].hist(seg_data[col], bins=20, alpha=0.7, label=segment, density=True)
                axes[i].set_title(col)
                axes[i].legend()
        
        plt.tight_layout()
        plt.show()

## 🏷️ Étape 4: Création de Personas et Nommage

Donnons des noms parlants à nos segments et définissons leurs personas.

In [ ]:
def creer_personas_automatique(df_segments):
    """Création automatique de personas basée sur les caractéristiques"""
    personas = {}
    
    for seg_id in sorted(set(labels_final)):
        if seg_id == -1:  # Ignorer le bruit DBSCAN
            continue
            
        seg_data = df_segments[df_segments['segment'] == seg_id]
        if len(seg_data) == 0:
            continue
        
        # Analyse des caractéristiques
        age_moy = seg_data['AGE'].mean() if 'AGE' in seg_data.columns else 35
        
        util_moy = seg_data['utilization'].mean() if 'utilization' in seg_data.columns else 0.5
        repay_moy = seg_data['repay_ratio_recent'].mean() if 'repay_ratio_recent' in seg_data.columns else 0.7
        retards_moy = seg_data['nb_retards'].mean() if 'nb_retards' in seg_data.columns else 1
        
        # Logique de nommage
        if util_moy > 0.6 and repay_moy > 0.8 and retards_moy < 0.5:
            nom = "🌟 Premium - Gros utilisateurs fiables"
            description = "Clients à forte valeur, utilisent beaucoup leur crédit et remboursent bien"
            
        elif retards_moy > 2 or repay_moy < 0.4:
            nom = "⚠️ Risque - Besoin d'accompagnement"
            description = "Clients avec difficultés de remboursement, nécessitent un suivi"
            
        elif util_moy < 0.2:
            nom = "😴 Dormants - Sous-utilisateurs"
            description = "Clients peu actifs, potentiel d'activation"
            
        elif age_moy < 30:
            nom = "🚀 Jeunes - Potentiel de croissance"
            description = "Jeunes clients, opportunité de développement"
            
        else:
            nom = "⚖️ Équilibrés - Cœur de cible"
            description = "Clients standards, base stable de l'activité"
        
        personas[seg_id] = {
            'nom': nom,
            'description': description,
            'taille': len(seg_data),
            'pourcentage': len(seg_data) / len(df_segments) * 100
        }
    
    return personas

personas = creer_personas_automatique(df_segments)

print("🎭 Personas créés:")
for seg_id, persona in personas.items():
    print(f"\n{persona['nom']}")
    print(f"  📝 {persona['description']}")
    print(f"  👥 {persona['taille']} clients ({persona['pourcentage']:.1f}%)")

## 💼 Étape 5: Recommandations Marketing Interactives

Définissons des actions concrètes pour chaque segment.

In [ ]:
@interact(
    segment=Dropdown(options=[f"Segment {i}: {personas[i]['nom']}" for i in personas.keys()], 
                    description='Segment:')
)
def recommandations_marketing(segment):
    """Recommandations marketing par segment"""
    seg_id = int(segment.split(':')[0].split()[-1])
    persona = personas[seg_id]
    seg_data = df_segments[df_segments['segment'] == seg_id]
    
    print(f"🎯 Plan d'action pour: {persona['nom']}")
    print(f"📊 Taille: {persona['taille']} clients ({persona['pourcentage']:.1f}%)")
    print(f"📝 Profil: {persona['description']}")
    
    # Recommandations spécifiques selon le type
    if "Premium" in persona['nom']:
        print(f"\n💎 Actions recommandées:")
        print(f"  1. 📈 Augmentation de limite de crédit (+20-30%)")
        print(f"  2. 🏆 Offres premium (carte gold/platinum)")
        print(f"  3. 🎁 Programme de fidélité exclusif")
        print(f"  4. 📞 Relation client dédiée")
        print(f"  5. 💰 Produits d'investissement/épargne")
        
        print(f"\n📊 KPI à suivre:")
        print(f"  • Taux d'acceptation des offres premium")
        print(f"  • Évolution du chiffre d'affaires par client")
        print(f"  • Taux de rétention")
        
    elif "Risque" in persona['nom']:
        print(f"\n🚨 Actions recommandées:")
        print(f"  1. 📞 Contact proactif pour accompagnement")
        print(f"  2. 📅 Plan de remboursement personnalisé")
        print(f"  3. 📚 Éducation financière")
        print(f"  4. 🔒 Révision des limites de crédit")
        print(f"  5. 🤝 Partenariat avec conseillers financiers")
        
        print(f"\n📊 KPI à suivre:")
        print(f"  • Réduction du taux de retard")
        print(f"  • Amélioration du score de remboursement")
        print(f"  • Taux de régularisation")
        
    elif "Dormants" in persona['nom']:
        print(f"\n🔄 Actions recommandées:")
        print(f"  1. 🎯 Campagne de réactivation ciblée")
        print(f"  2. 💳 Offres cashback attractives")
        print(f"  3. 🛍️ Partenariats marchands (e-commerce)")
        print(f"  4. 📱 Notifications push personnalisées")
        print(f"  5. 🎁 Bonus d'activation temporaire")
        
        print(f"\n📊 KPI à suivre:")
        print(f"  • Taux d'activation")
        print(f"  • Évolution de l'utilisation")
        print(f"  • Fréquence des transactions")
        
    elif "Jeunes" in persona['nom']:
        print(f"\n🚀 Actions recommandées:")
        print(f"  1. 📱 Expérience mobile optimisée")
        print(f"  2. 🎓 Offres étudiants/jeunes actifs")
        print(f"  3. 💡 Éducation financière gamifiée")
        print(f"  4. 🌐 Marketing digital (réseaux sociaux)")
        print(f"  5. 🏠 Produits logement (prêt immobilier)")
        
        print(f"\n📊 KPI à suivre:")
        print(f"  • Taux de conversion des offres")
        print(f"  • Engagement digital")
        print(f"  • Évolution vers segments premium")
        
    else:  # Équilibrés
        print(f"\n⚖️ Actions recommandées:")
        print(f"  1. 📧 Communication régulière et équilibrée")
        print(f"  2. 🔄 Cross-selling modéré (assurance, épargne)")
        print(f"  3. 💬 Enquêtes de satisfaction")
        print(f"  4. 🎯 Offres saisonnières")
        print(f"  5. 🤝 Maintien de la relation")
        
        print(f"\n📊 KPI à suivre:")
        print(f"  • Taux de satisfaction")
        print(f"  • Stabilité du portefeuille")
        print(f"  • Taux de cross-selling")

## 💰 Étape 6: Estimation d'Impact Business

Estimons le potentiel de revenus de chaque segment.

In [ ]:
def calculer_impact_business(df_segments, personas):
    """Calcule l'impact business estimé par segment"""
    
    impacts = []
    
    for seg_id, persona in personas.items():
        seg_data = df_segments[df_segments['segment'] == seg_id]
        taille = len(seg_data)
        
        # Estimation du revenu moyen actuel par client (proxy)
        if 'facture_moyenne' in seg_data.columns:
            revenu_actuel = seg_data['facture_moyenne'].mean() * 0.02  # 2% de marge estimée
        else:
            revenu_actuel = 100  # Valeur par défaut
        
        # Potentiel d'amélioration selon le segment
        if "Premium" in persona['nom']:
            uplift_potentiel = 0.25  # +25% via upselling
            cout_acquisition = revenu_actuel * 0.1  # 10% du revenu
            
        elif "Risque" in persona['nom']:
            uplift_potentiel = -0.05  # Réduction des pertes de 5%
            cout_acquisition = revenu_actuel * 0.15  # Coût de suivi
            
        elif "Dormants" in persona['nom']:
            uplift_potentiel = 0.15  # +15% via réactivation
            cout_acquisition = revenu_actuel * 0.08
            
        elif "Jeunes" in persona['nom']:
            uplift_potentiel = 0.20  # +20% via développement
            cout_acquisition = revenu_actuel * 0.12
            
        else:  # Équilibrés
            uplift_potentiel = 0.05  # +5% via cross-selling
            cout_acquisition = revenu_actuel * 0.05
        
        # Calculs
        revenu_total_actuel = taille * revenu_actuel
        gain_potentiel = revenu_total_actuel * uplift_potentiel
        cout_total = taille * cout_acquisition
        roi = (gain_potentiel - cout_total) / cout_total if cout_total > 0 else 0
        
        impacts.append({
            'Segment': persona['nom'],
            'Clients': taille,
            'Revenu actuel (k€)': revenu_total_actuel / 1000,
            'Gain potentiel (k€)': gain_potentiel / 1000,
            'Coût actions (k€)': cout_total / 1000,
            'ROI': roi,
            'Priorité': 'Haute' if roi > 2 else 'Moyenne' if roi > 1 else 'Faible'
        })
    
    return pd.DataFrame(impacts).round(1)

# Calcul et affichage
df_impact = calculer_impact_business(df_segments, personas)
print("💰 Estimation d'impact business par segment:")
display(df_impact)

# Visualisation
plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.bar(range(len(df_impact)), df_impact['Gain potentiel (k€)'])
plt.title('Gain potentiel par segment')
plt.xticks(range(len(df_impact)), [f"S{i}" for i in range(len(df_impact))], rotation=45)
plt.ylabel('Gain (k€)')

plt.subplot(2, 2, 2)
plt.bar(range(len(df_impact)), df_impact['ROI'])
plt.title('ROI par segment')
plt.xticks(range(len(df_impact)), [f"S{i}" for i in range(len(df_impact))], rotation=45)
plt.ylabel('ROI')
plt.axhline(y=1, color='red', linestyle='--', label='Seuil rentabilité')
plt.legend()

plt.subplot(2, 2, 3)
sizes = df_impact['Clients']
plt.pie(sizes, labels=[f"S{i}" for i in range(len(df_impact))], autopct='%1.1f%%')
plt.title('Répartition des clients')

plt.subplot(2, 2, 4)
colors = ['green' if p == 'Haute' else 'orange' if p == 'Moyenne' else 'red' for p in df_impact['Priorité']]
plt.bar(range(len(df_impact)), df_impact['Gain potentiel (k€)'], color=colors)
plt.title('Priorisation des segments')
plt.xticks(range(len(df_impact)), [f"S{i}" for i in range(len(df_impact))], rotation=45)
plt.ylabel('Gain (k€)')

plt.tight_layout()
plt.show()

## 📋 Étape 7: Plan d'Action Final

Synthèse et roadmap de mise en œuvre.

In [ ]:
print("🎯 PLAN D'ACTION - SEGMENTATION CLIENTS CRÉDIT")
print("=" * 50)

print(f"\n📊 RÉSUMÉ EXÉCUTIF:")
print(f"• Algorithme retenu: {modele_final}")
print(f"• Nombre de segments: {len(personas)}")
print(f"• Gain potentiel total: {df_impact['Gain potentiel (k€)'].sum():.0f}k€")
print(f"• ROI moyen: {df_impact['ROI'].mean():.1f}")

print(f"\n🏆 TOP 3 SEGMENTS PRIORITAIRES:")
top_segments = df_impact.nlargest(3, 'ROI')
for i, (_, row) in enumerate(top_segments.iterrows(), 1):
    print(f"{i}. {row['Segment']} (ROI: {row['ROI']:.1f}, Gain: {row['Gain potentiel (k€)']:.0f}k€)")

print(f"\n📅 ROADMAP DE MISE EN ŒUVRE:")
print(f"📍 Phase 1 (0-3 mois): Segments haute priorité")
print(f"  • Mise en place des actions pour les segments ROI > 2")
print(f"  • Définition des KPI et tableaux de bord")
print(f"  • Formation des équipes commerciales")

print(f"\n📍 Phase 2 (3-6 mois): Extension et optimisation")
print(f"  • Déploiement sur segments priorité moyenne")
print(f"  • Analyse des premiers résultats")
print(f"  • Ajustement des stratégies")

print(f"\n📍 Phase 3 (6-12 mois): Industrialisation")
print(f"  • Automatisation des campagnes")
print(f"  • Mise à jour périodique des segments")
print(f"  • Extension à d'autres produits")

print(f"\n🔧 PRÉREQUIS TECHNIQUES:")
print(f"• Mise à jour du CRM avec les segments")
print(f"• Paramétrage des campagnes marketing")
print(f"• Formation des équipes")
print(f"• Mise en place du reporting")

print(f"\n📊 SUIVI ET MESURE:")
print(f"• Révision mensuelle des KPI par segment")
print(f"• Recalcul trimestriel des segments")
print(f"• Bilan annuel et optimisation de la stratégie")

# Export des résultats
df_segments.to_csv('segments_clients_final.csv', index=False)
df_impact.to_csv('impact_business_segments.csv', index=False)

print(f"\n💾 FICHIERS EXPORTÉS:")
print(f"• segments_clients_final.csv (données + segments)")
print(f"• impact_business_segments.csv (analyse d'impact)")

## ✅ Récapitulatif Final - Partie 3

🎉 **Félicitations!** Vous avez terminé l'analyse complète de segmentation clients.

**Ce que vous avez accompli:**
- ✅ Sélection du meilleur algorithme de clustering
- ✅ Création de personas détaillés par segment  
- ✅ Définition d'actions marketing concrètes
- ✅ Estimation de l'impact business et ROI
- ✅ Élaboration d'un plan d'action opérationnel

**Livrables créés:**
- 📊 Segments clients avec personas
- 💼 Recommandations marketing par segment
- 💰 Analyse d'impact business et ROI
- 📋 Roadmap de mise en œuvre
- 📁 Fichiers d'export pour les équipes opérationnelles

**Prochaines étapes recommandées:**
1. Présentation des résultats au comité de direction
2. Validation du budget et des ressources
3. Lancement de la Phase 1 (segments haute priorité)
4. Mise en place du suivi et des KPI

🚀 **Votre segmentation est prête à être déployée!**